# 动手练习：症状网络与网络精神病学

来自《生物医学网络科学》（Biomedical Network Science）。 参考答案版。

## 任务 1：相关网络与偏相关网络

本练习前 5 个任务使用 numpy 从一个已知的“真实”偏相关网络生成的**模拟数据**，只用于演示方法，不代表任何真实人群的抑郁症状结构。真实量表条目是 0–3 分的有序变量，这里简化为连续变量；真实研究通常用 R（qgraph、bootnet、NetworkComparisonTest）或 JASP 完成，本练习只用 numpy 演示背后的原理。

9 个症状按 PHQ-9 条目顺序：兴趣减退、情绪低落、睡眠问题、疲乏、食欲改变、自我评价低、注意力差、精神运动改变、自杀意念；真实网络有 10 条边（`TRUE_EDGES`）。

由样本相关矩阵 `R` 求精度矩阵 $K=R^{-1}$，计算偏相关矩阵 `P`（$P_{ij}=-K_{ij}/\sqrt{K_{ii}K_{jj}}$，对角线设为 0）；再统计 $|r|>0.1$ 的边数 `n_r` 与 $|p|>0.1$ 的边数 `n_p`。

期望看到：相关网络的边明显多于 10 条（间接关联也被画成了边），偏相关网络更接近真实结构。例如“睡眠问题–注意力差”在真实网络中没有直接边（只经由“疲乏”相连），$r$ 不小，偏相关却接近 0。

In [ ]:
# 准备（直接运行）
import numpy as np
import matplotlib.pyplot as plt

names = ["兴趣减退", "情绪低落", "睡眠问题", "疲乏", "食欲改变",
         "自我评价低", "注意力差", "精神运动改变", "自杀意念"]
TRUE_EDGES = {(0, 1): .40, (1, 5): .30, (1, 8): .25, (5, 8): .25, (2, 3): .40,
              (3, 6): .30, (3, 7): .20, (3, 4): .20, (0, 3): .20, (6, 7): .20}


def simulate_symptoms(n, seed=0):
    """模拟数据，仅用于演示方法：从已知偏相关结构的多元正态分布抽样。"""
    K = np.eye(9)
    for (i, j), w in TRUE_EDGES.items():
        K[i, j] = K[j, i] = -w          # 精度矩阵负的非对角元 → 正偏相关
    rng = np.random.default_rng(seed)
    L = np.linalg.cholesky(np.linalg.inv(K))   # Cholesky 抽样：各平台（含浏览器）得到完全相同的数据
    return rng.standard_normal((n, 9)) @ L.T


X = simulate_symptoms(500)
R = np.corrcoef(X, rowvar=False)
iu = np.triu_indices(9, k=1)

In [ ]:
def pcor_from_corr(R):
    K = np.linalg.inv(R)
    d = np.sqrt(np.diag(K))
    P = -K / np.outer(d, d)
    np.fill_diagonal(P, 0)
    return P


P = pcor_from_corr(R)
n_r = int(np.sum(np.abs(R[iu]) > 0.1))
n_p = int(np.sum(np.abs(P[iu]) > 0.1))
print("真实边 10 条；|r| > 0.1 的边 %d 条；|pcor| > 0.1 的边 %d 条" % (n_r, n_p))
print("睡眠问题–注意力差：r = %.2f，pcor = %.2f" % (R[2, 6], P[2, 6]))

In [ ]:
# 核对（不用修改）
assert P is not None and P.shape == (9, 9), "P 应为 9×9 矩阵"
assert np.allclose(np.diag(P), 0), "P 的对角线应设为 0"
assert P[0, 1] > 0.3, "兴趣减退–情绪低落的偏相关应约为 0.4 且为正；检查公式前的负号"
# 期望值按同一份数据现算（多元正态抽样在不同平台上的随机数略有差别，不写死）
_K = np.linalg.inv(R); _d = np.sqrt(np.diag(_K)); _P = -_K / np.outer(_d, _d)
assert n_r == int(np.sum(np.abs(R[iu]) > 0.1)), "n_r 应等于相关矩阵上三角中 |r| > 0.1 的个数"
assert n_p == int(np.sum(np.abs(_P[iu]) > 0.1)), "n_p 应等于偏相关矩阵上三角中 |pcor| > 0.1 的个数"
assert n_p < n_r, "偏相关去掉了间接关联，|pcor| > 0.1 的边应比 |r| > 0.1 的少"
assert abs(P[2, 6]) < 0.1 < R[2, 6], "睡眠–注意力：相关不小，偏相关应接近 0"
print("✓ 任务 1 通过")

## 任务 2：强度中心性与预期影响

写函数 `centrality(W)`，返回 `(strength, ei)`：强度 $s_i=\sum_j|w_{ij}|$，预期影响 $\mathrm{EI}_i=\sum_j w_{ij}$。

1. 对任务 1 的偏相关网络 `P` 计算 `strength`、`ei`，找出强度最高的 3 个症状的下标 `top3`（从高到低）；
2. 对 setup 中带一条负边的小网络 `W_neg` 计算 `s_neg`、`ei_neg`。

期望看到：疲乏、情绪低落强度最高（与真实结构一致）；本例偏相关网络几乎没有负边，所以 EI 与强度接近；有负边时二者才明显不同。

In [ ]:
# 准备（直接运行）
W_neg = np.array([[0, 0.30, 0.20, -0.15],
                  [0.30, 0, 0.10, 0],
                  [0.20, 0.10, 0, 0.05],
                  [-0.15, 0, 0.05, 0]])

In [ ]:
def centrality(W):
    return np.abs(W).sum(1), W.sum(1)


strength, ei = centrality(P)
top3 = [int(i) for i in np.argsort(strength)[::-1][:3]]
s_neg, ei_neg = centrality(W_neg)
print("强度最高的 3 个症状：", [names[i] for i in top3])
print("小网络节点 0：强度 %.2f，EI %.2f" % (s_neg[0], ei_neg[0]))

In [ ]:
# 核对（不用修改）
assert strength is not None and len(strength) == 9, "strength 应有 9 个值"
assert top3 is not None and top3[:2] == [3, 1], "强度最高的应依次是疲乏（3）和情绪低落（1）"
assert abs(s_neg[0] - 0.65) < 1e-9, "节点 0 的强度 = 0.30 + 0.20 + 0.15 = 0.65（取绝对值）"
assert abs(ei_neg[0] - 0.35) < 1e-9, "节点 0 的预期影响 = 0.30 + 0.20 − 0.15 = 0.35（保留符号）"
print("✓ 任务 2 通过")

## 任务 3：偏相关也会“制造”关联：对撞偏倚

模拟一个对撞结构 $A\to C\leftarrow B$：$A$、$B$ 相互独立，$C=A+B+\varepsilon$（2000 人，用 setup 中的 `rng3`；模拟数据）。计算 $A$、$B$ 的皮尔逊相关 `r_ab` 和控制 $C$ 之后的偏相关 `pc_ab`。

期望看到：`r_ab` ≈ 0，`pc_ab` ≈ −0.5。偏相关网络会在 A 与 B 之间画出一条（负的）边，而它们之间并没有直接作用。例如“失眠”和“工作压力”都导致“疲乏”时，控制疲乏后二者可能呈负偏相关。

> **提示：** 把三列拼成矩阵，用任务 1 的方法求偏相关矩阵。

In [ ]:
# 准备（直接运行）
rng3 = np.random.default_rng(4)
a_ = rng3.standard_normal(2000)
b_ = rng3.standard_normal(2000)
c_ = a_ + b_ + rng3.standard_normal(2000)

In [ ]:
M = np.corrcoef(np.vstack([a_, b_, c_]))
r_ab = M[0, 1]
K3 = np.linalg.inv(M)
pc_ab = -K3[0, 1] / np.sqrt(K3[0, 0] * K3[1, 1])
print("r_AB = %.3f；控制 C 后的偏相关 = %.3f" % (r_ab, pc_ab))

In [ ]:
# 核对（不用修改）
assert r_ab is not None and abs(r_ab) < 0.06, "A、B 相互独立，r_AB 应接近 0"
assert pc_ab is not None and -0.6 < pc_ab < -0.45, "控制共同结果 C 后，A、B 的偏相关应约为 −0.5（负号来自对撞）"
print("✓ 任务 3 通过")

## 任务 4：中心性稳定吗？简化版病例删除 bootstrap

写函数 `case_drop(X, keep_props, n_boot, seed)`：对每个保留比例 $p$，随机抽取 $p\times n$ 个被试（不放回），重新计算偏相关网络的强度，与全样本强度做相关；返回 `{p: 这些相关的第 5 百分位}`。

CS 系数 = 第 5 百分位仍 ≥ 0.7 时可以删去的最大比例（$1-p$）。对 $n=300$ 和 $n=100$ 的模拟数据（`seed=2`）分别得到结果字典 `res300`、`res100` 和 CS 系数 `cs300`、`cs100`。setup 提供了 `pcor_from_corr(R)`。

期望看到：保留比例越低，相关越低；$n=100$ 时 CS 明显下降——小样本的中心性排序不可靠（Epskamp 等建议 CS 至少 0.25，最好高于 0.5）。

In [ ]:
# 准备（直接运行）
keep_props = (0.9, 0.7, 0.5, 0.3)


def pcor_from_corr(R):
    """由相关矩阵求偏相关矩阵（对角线为 0）。"""
    K = np.linalg.inv(R)
    d = np.sqrt(np.diag(K))
    P = -K / np.outer(d, d)
    np.fill_diagonal(P, 0)
    return P

In [ ]:
def case_drop(X, keep_props=keep_props, n_boot=100, seed=1):
    rng = np.random.default_rng(seed)
    s_full = np.abs(pcor_from_corr(np.corrcoef(X, rowvar=False))).sum(axis=1)
    n = X.shape[0]
    out = {}
    for p in keep_props:
        cors = []
        for _ in range(n_boot):
            idx = rng.choice(n, int(p * n), replace=False)
            s_sub = np.abs(pcor_from_corr(np.corrcoef(X[idx], rowvar=False))).sum(axis=1)
            cors.append(np.corrcoef(s_sub, s_full)[0, 1])
        out[p] = np.quantile(cors, 0.05)
    return out


def cs_coef(res):
    return max([round(1 - p, 2) for p, q in res.items() if q >= 0.7], default=0.0)


res300 = case_drop(simulate_symptoms(300, seed=2))
res100 = case_drop(simulate_symptoms(100, seed=2))
cs300, cs100 = cs_coef(res300), cs_coef(res100)
print("n = 300：CS ≈ %.1f" % cs300, {p: round(float(q), 2) for p, q in res300.items()})
print("n = 100：CS ≈ %.1f" % cs100, {p: round(float(q), 2) for p, q in res100.items()})

In [ ]:
# 核对（不用修改）
assert cs300 is not None and cs100 is not None and res300 is not None, "res300、cs300、cs100 还没有计算"
assert len(res300) == 4 and res300[0.9] > res300[0.3], "保留比例越低，强度与全样本的相关应越低"
assert 0.3 <= cs300 <= 0.7, "n = 300 时 CS 应约为 0.5"
assert cs100 < cs300 and cs100 <= 0.3, "n = 100 时 CS 应明显低于 n = 300"
print("✓ 任务 4 通过")

## 任务 5：EBICglasso：用 EBIC 选择惩罚强度

setup 给出图形 LASSO 函数 `glasso(S, lam, W0)`（块坐标下降，对角元不惩罚，返回精度矩阵估计和协方差估计）。写函数 `ebic_glasso(X, gamma=0.5, n_lambda=40)`：

1. $S$ = 样本相关矩阵，$\lambda_{\max}=\max_{i\ne j}|S_{ij}|$，在 $[0.01\lambda_{\max},\lambda_{\max}]$ 上取 40 个对数等距的 $\lambda$，从大到小依次拟合（把上一个 $W$ 作为热启动）；
2. 对每个解计算 $\mathrm{EBIC}=-n\,[\log\det K-\operatorname{tr}(SK)]+E\log n+4\gamma E\log p$，$E$ 为非零边数；
3. 返回 EBIC 最小者对应的偏相关矩阵（对角线 0，$|p|<10^{-8}$ 置 0）和 $\lambda$。

对 $n=500$ 的模拟数据（`seed=3`）得到 `P_glasso`，统计非零边数 `n_edges`、命中的真实边数 `hits`、真实边的平均估计权重 `mean_true_w`。

期望看到：真实边全部命中，只多出少量弱的虚假边；但真实边的平均权重低于真值的平均 0.27——收缩偏倚。

In [ ]:
# 准备（直接运行）
def glasso(S, lam, W0=None, max_iter=200, tol=1e-5):
    q = S.shape[0]
    W = S.copy() if W0 is None else W0.copy()
    B = np.zeros((q, q))
    for _ in range(max_iter):
        W_old = W.copy()
        for j in range(q):
            idx = np.arange(q) != j
            W11, s12, beta = W[np.ix_(idx, idx)], S[idx, j], B[idx, j].copy()
            for _ in range(100):
                beta_old = beta.copy()
                for k in range(q - 1):
                    r = s12[k] - W11[k] @ beta + W11[k, k] * beta[k]
                    beta[k] = np.sign(r) * max(abs(r) - lam, 0) / W11[k, k]
                if np.max(np.abs(beta - beta_old)) < tol:
                    break
            B[idx, j] = beta
            W[idx, j] = W[j, idx] = W11 @ beta
        if np.mean(np.abs(W - W_old)) < tol:
            break
    Theta = np.zeros((q, q))
    for j in range(q):
        idx = np.arange(q) != j
        t22 = 1.0 / (W[j, j] - W[idx, j] @ B[idx, j])
        Theta[j, j] = t22
        Theta[idx, j] = -B[idx, j] * t22
    return (Theta + Theta.T) / 2, W


X500 = simulate_symptoms(500, seed=3)

In [ ]:
def ebic_glasso(X, gamma=0.5, n_lambda=40, ratio=0.01):
    n, q = X.shape
    S = np.corrcoef(X, rowvar=False)
    lam_max = np.max(np.abs(S - np.eye(q)))
    best, W = None, None
    for lam in np.exp(np.linspace(np.log(lam_max), np.log(ratio * lam_max), n_lambda)):
        K, W = glasso(S, lam, W0=W)
        E = int(np.sum(np.abs(K[np.triu_indices(q, 1)]) > 1e-8))
        loglik = np.linalg.slogdet(K)[1] - np.trace(S @ K)
        ebic = -n * loglik + E * np.log(n) + 4 * gamma * E * np.log(q)
        if best is None or ebic < best[0]:
            best = (ebic, lam, K)
    _, lam, K = best
    d = np.sqrt(np.diag(K))
    Pg = -K / np.outer(d, d)
    np.fill_diagonal(Pg, 0)
    Pg[np.abs(Pg) < 1e-8] = 0
    return Pg, lam


P_glasso, lam_sel = ebic_glasso(X500)
est = {(i, j) for i, j in zip(*iu) if P_glasso[i, j] != 0}
n_edges = len(est)
hits = len(est & set(TRUE_EDGES))
mean_true_w = float(np.mean([P_glasso[e] for e in TRUE_EDGES]))
print("λ = %.3f；非零边 %d 条，命中真实边 %d/10；真实边平均权重 %.3f（真值平均 0.27）" % (lam_sel, n_edges, hits, mean_true_w))

In [ ]:
# 核对（不用修改）
assert P_glasso is not None, "P_glasso 还没有计算"
assert np.allclose(P_glasso, P_glasso.T), "偏相关矩阵应对称"
assert hits == 10, "n = 500 时 10 条真实边都应被选中"
assert 10 <= n_edges <= 16, "非零边数应略多于 10（约 13 条）；若接近 36 条，检查 EBIC 的复杂度惩罚项"
assert mean_true_w < 0.27, "正则化会把边权压小：真实边的平均估计权重应低于真值 0.27"
print("✓ 任务 5 通过")

## 任务 6：小分析：真实 PHQ-9 数据的亚组网络

用真实数据 `data/nhanes_phq9_2017_2018.csv`（美国 NHANES 2017–2018，5,068 名成人，公有领域；出处见本讲真实数据案例与 `data/SOURCES.md`）。列 `PHQ1`–`PHQ9` 为 9 个条目（0–3 分），`sex`（1 男，2 女），`age`（岁）。

自己定义一个亚组 `mask`（例如 60 岁及以上，或女性），用 Spearman 相关矩阵求**未正则化**的偏相关网络 `P_sub`，计算强度 `strength_sub`，并找出强度最高的条目 `top_item`（列名，如 `"PHQ2"`）。

和全体样本、和另一个亚组比一比：强度最高的条目一样吗？亚组越小，你觉得这个排序有多可信——该怎样检验（提示：任务 4 的病例删除 bootstrap，或网络比较检验）？

In [ ]:
# 准备（直接运行）
import pandas as pd
from scipy.stats import spearmanr

phq = pd.read_csv("data/nhanes_phq9_2017_2018.csv")
item_cols = ["PHQ%d" % i for i in range(1, 10)]

In [ ]:
mask = phq.age >= 60
Xs = phq.loc[mask, item_cols].to_numpy(float)
P_sub = pcor_from_corr(spearmanr(Xs).statistic)
strength_sub = np.abs(P_sub).sum(1)
top_item = item_cols[int(np.argmax(strength_sub))]
print("亚组 %d 人；强度最高的条目 %s；强度 %s" % (mask.sum(), top_item, np.round(strength_sub, 2).tolist()))

In [ ]:
# 核对（不用修改）
assert int(np.sum(mask)) >= 100, "亚组太小（少于 100 人）时偏相关网络很不稳定，请换一个更大的亚组"
assert P_sub is not None and np.shape(P_sub) == (9, 9), "P_sub 应为 9×9"
assert np.allclose(P_sub, np.asarray(P_sub).T) and np.all(np.abs(np.asarray(P_sub)) <= 1), "偏相关矩阵应对称，元素在 −1 到 1 之间"
assert strength_sub is not None and len(strength_sub) == 9 and np.all(np.asarray(strength_sub) >= 0), "strength_sub 应是 9 个非负值"
assert top_item in item_cols, "top_item 应是 PHQ1–PHQ9 中的一个列名"
print("✓ 任务 6 通过")